# MOSFET Loss Equations and Assumptions

This notebook records the final electrical-loss equations, numerical assumptions and modelling boundaries used for the IRFZ44N MOSFET.

It is intended as a compact reference for the analytical electrical model. Detailed calculations, sensitivity studies and LTspice waveform processing are contained in `mosfet_loss_calculation_main.ipynb`.

## 1. Frozen Electrical Inputs

| Parameter | Value |
|---|---:|
| Input voltage, Vin | 24 V |
| Output voltage, Vout | 12 V |
| Output current used for baseline comparison | 10 A |
| Switching frequency, fs | 50 kHz |
| Duty cycle, D | 0.5 |
| MOSFET | IRFZ44N |
| Gate-drive voltage | 10 V |
| Rise time, tr | 60 ns |
| Fall time, tf | 45 ns |
| LTspice-matched Rds(on) | 0.0139 ohm |
| Datasheet maximum Rds(on) at 25°C | 0.0175 ohm |
| LTspice simulation temperature | 25°C |

Two Rds(on) values are intentionally retained:

- **0.0139 ohm** is used for direct analytical comparison with the LTspice MOSFET model.
- **0.0175 ohm** is the datasheet maximum at 25°C and is used as the conservative reference for thermal and electro-thermal analysis.

## 2. Conduction-Loss Equation

For the buck-converter MOSFET:

`Pcond = I² × Rds(on) × D`

This is equivalent to `I_MOSFET,RMS² × Rds(on)` when the MOSFET conducts the load current for a fraction `D` of each switching period.

At **10 A** and **D = 0.5**:

- Using **Rds(on) = 0.0139 ohm**: `Pcond = 0.695 W`
- Using **Rds(on) = 0.0175 ohm**: `Pcond = 0.875 W`

**Analytical simplification:** nominal load current is used in the conduction-loss expression rather than the exact MOSFET RMS current, so inductor-current ripple is neglected in this first-order loss model. This assumption is documented rather than treated as exact.

## 3. Switching-Loss Equation

Switching loss is estimated using:

`Psw = 0.5 × VDS × ID × (tr + tf) × fs`

Using:

- `VDS = 24 V`
- `ID = 10 A`
- `tr = 60 ns`
- `tf = 45 ns`
- `fs = 50 kHz`

gives:

**Psw = 0.630 W**

This simplified switching-loss estimate assumes approximately linear voltage-current overlap during the switching transitions.

## 4. Total Analytical MOSFET Loss

`Ptotal = Pcond + Psw`

At the 10 A baseline operating point:

| Case | Conduction loss (W) | Switching loss (W) | Total loss (W) |
|---|---:|---:|---:|
| LTspice-matched Rds(on) = 0.0139 ohm | 0.695 | 0.630 | **1.325** |
| Datasheet maximum Rds(on) = 0.0175 ohm | 0.875 | 0.630 | **1.505** |

The **1.505 W** result is retained as the conservative 25°C baseline heat input for thermal modelling.

## 5. LTspice Validation Reference

The average MOSFET power obtained from the exported LTspice voltage and current waveforms at the 10 A operating point was approximately:

**PLTspice = 0.731 W**

The direct analytical-to-LTspice comparison uses the **0.0139 ohm** analytical case because it more closely matches the resistance represented by the LTspice device model.

The waveform-based LTspice result is approximately **44.8% lower than the analytical estimate when the difference is normalised to the analytical value**. Equivalently, the analytical estimate is about 81.3% higher than the LTspice result when normalised to LTspice. The project uses the first convention consistently and retains the discrepancy as a modelling uncertainty rather than forcing the two methods to agree.

## 6. Main Modelling Assumptions

- The buck converter is treated using the nominal 24 V to 12 V operating condition with duty cycle 0.5.
- MOSFET conduction loss is estimated from load current, duty cycle and Rds(on).
- Switching transitions are represented by the simplified linear-overlap switching-loss equation.
- Datasheet rise and fall times may not exactly match the switching behaviour of the LTspice model.
- The LTspice gate source is ideal, so simulated transitions may differ from a practical gate-driver circuit.
- Gate-driver loss, PCB resistance, stray inductance, diode loss, inductor loss and capacitor ESR are not included in the MOSFET-loss calculation.
- The quoted efficiency elsewhere in the project is therefore a **MOSFET-loss-based efficiency**, not full converter efficiency.
- The 10 W and 15 W thermal-stress cases used later in the project are imposed heat loads and are not electrically derived from these equations.

## 7. Model Handoff

The conservative **0.0175 ohm** resistance is used in the temperature-dependent Rds(on) model and in the electro-thermal coupling notebook.

The electrical operating-point study remains based on **5 A, 10 A and 20 A**, while the **10 W and 15 W** cases are treated separately as thermal-stress cases.
